# ASSISTments: Random Forest de la práctica

Carga la tabla preparada, separa `X` y `y`, ajusta el Random Forest de la práctica y calcula las métricas solicitadas. La partición `source_split` ya está definida en los datos.

La tabla reporta solo accuracy, precisión macro, recall macro y F1 macro para entrenamiento y evaluación. Las gráficas de distribución, métricas, matriz de confusión, ROC e importancia de variables se conservan.


## 1. Entrada: CSV preparado

Sube `assistments_rf_dataset.csv.gz` desde Colab. La preparación de los archivos fuente se realiza en el notebook anterior.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from google.colab import files

SEED = 42
N_ESTIMATORS = 600
files.upload()  # Selecciona assistments_rf_dataset.csv.gz
DATASET_CSV = Path('/content/assistments_rf_dataset.csv.gz')
df = pd.read_csv(DATASET_CSV, low_memory=False)

print(f'{len(df):,} filas | {df.student_id.nunique():,} estudiantes | {df.shape[1]:,} columnas')
display(df['source_split'].value_counts().rename_axis('partición').to_frame('filas'))
display(df['score'].value_counts().sort_index().rename_axis('score').to_frame('filas'))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='score', color='#4C78A8')
plt.title('Distribución del resultado real')
plt.xlabel('score: 0 = no exitoso, 1 = exitoso')
plt.ylabel('Cantidad de problemas')
plt.tight_layout()
plt.show()


## 2. Preprocesamiento, modelado y evaluación

`score` se separa como variable dependiente `y`; las otras 722 columnas son `X`. Se conservan las particiones oficiales. El Random Forest se ajusta con train y se evalúa con evaluation.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
)

# score es y; IDs y partición se conservan en df, pero no entran en X.
id_columns = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
feature_columns = [c for c in df.columns if c not in id_columns]
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
y = df['score'].astype('int8')

# Se usa directamente la partición oficial del dataset.
train_mask = df['source_split'].eq('train')
eval_mask = df['source_split'].eq('evaluation')
X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[eval_mask].reset_index(drop=True)
y_train = y.loc[train_mask].to_numpy()
y_test = y.loc[eval_mask].to_numpy()

rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=19,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)

pred_train = rf.predict(X_train)
y_pred = rf.predict(X_test)
probabilities = rf.predict_proba(X_test)
class_index = {int(label): i for i, label in enumerate(rf.classes_)}
y_prob_non_success = probabilities[:, class_index[0]]
y_non_success = (y_test == 0).astype('int8')

metric_names = ['Accuracy', 'Precisión macro*', 'Recall macro*', 'F1 macro*']
metrics_train = {
    'Accuracy': accuracy_score(y_train, pred_train),
    'Precisión macro*': precision_score(y_train, pred_train, average='macro'),
    'Recall macro*': recall_score(y_train, pred_train, average='macro'),
    'F1 macro*': f1_score(y_train, pred_train, average='macro')
}
metrics_eval = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precisión macro*': precision_score(y_test, y_pred, average='macro'),
    'Recall macro*': recall_score(y_test, y_pred, average='macro'),
    'F1 macro*': f1_score(y_test, y_pred, average='macro')
}
metrics_table = pd.DataFrame([
    {'Partición': 'Entrenamiento', **metrics_train},
    {'Partición': 'Evaluación (prueba)', **metrics_eval}
]).set_index('Partición')
display(metrics_table.round(4))
print('* Promedio macro: mismo peso para score=0 y score=1.')

parameters_used = {
    'n_estimators': rf.n_estimators,
    'max_depth': rf.max_depth,
    'min_samples_split': rf.min_samples_split,
    'min_samples_leaf': rf.min_samples_leaf,
    'max_features': rf.max_features,
    'bootstrap': rf.bootstrap,
    'random_state': rf.random_state,
    'n_jobs': rf.n_jobs
}
display(pd.Series(parameters_used, name='valor').to_frame())


## 3. Gráficas de evaluación

Se muestran las cuatro métricas acordadas para train y evaluation, la matriz de confusión y ROC para `score=0`, y las 15 variables de mayor importancia. El AUC aparece solo como referencia dentro del gráfico ROC; no forma parte de la tabla de métricas.


In [ ]:
# Gráfico comparativo: solo las cuatro métricas solicitadas.
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(metric_names))
width = 0.36
train_values = [metrics_train[name] for name in metric_names]
eval_values = [metrics_eval[name] for name in metric_names]
train_bars = ax.bar(x - width/2, train_values, width, label='Entrenamiento', color='#4C78A8')
eval_bars = ax.bar(x + width/2, eval_values, width, label='Evaluación', color='#F58518')
ax.bar_label(train_bars, fmt='%.3f', padding=2, fontsize=8)
ax.bar_label(eval_bars, fmt='%.3f', padding=2, fontsize=8)
ax.set_xticks(x, metric_names, rotation=10)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Valor (0 a 1)')
ax.set_title('Métricas solicitadas: entrenamiento y evaluación')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# Matriz de confusión y curva ROC de la clase score=0.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No exitoso (0)', 'Exitoso (1)']
).plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='d')
axes[0].set_title('Matriz de confusión · evaluación')

fpr, tpr, _ = roc_curve(y_non_success, y_prob_non_success)
auc_score0 = roc_auc_score(y_non_success, y_prob_non_success)
axes[1].plot(fpr, tpr, color='#F58518', label=f'AUC = {auc_score0:.3f}')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel('Tasa de falsos positivos')
axes[1].set_ylabel('Tasa de verdaderos positivos')
axes[1].legend(loc='lower right')
axes[1].set_title('Curva ROC: score=0')
plt.tight_layout()
plt.show()

importance = pd.Series(rf.feature_importances_, index=feature_columns).sort_values(ascending=False)
top_features = importance.head(15).sort_values()
plt.figure(figsize=(9, 6))
top_features.plot(kind='barh', color='#54A24B')
plt.title('15 variables más importantes en Random Forest')
plt.xlabel('Importancia relativa')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()


## 4. Predicción por lote con un CSV nuevo (opcional)

Esta celda procesa un archivo nuevo ya transformado con el mismo esquema de predictores. La celda está desactivada por defecto; se activa únicamente para procesar un CSV nuevo.


In [ ]:
EJECUTAR_PREDICCION_REAL = False  # Activa solo al disponer de un CSV nuevo ya procesado.

if EJECUTAR_PREDICCION_REAL:
    archivos = files.upload()
    nombre = next(iter(archivos))
    ruta = Path('/content') / nombre
    ruta.write_bytes(archivos[nombre])
    nuevos_df = pd.read_csv(ruta, low_memory=False)

    columnas_faltantes = sorted(set(feature_columns) - set(nuevos_df.columns))
    if columnas_faltantes:
        raise ValueError(f'Faltan variables del esquema de entrenamiento: {columnas_faltantes[:10]}')

    X_nuevo = nuevos_df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
    pred_nuevas = rf.predict(X_nuevo)
    prob_nuevas = rf.predict_proba(X_nuevo)
    columnas_id = [c for c in ['student_id', 'unit_test_assignment_log_id', 'problem_id'] if c in nuevos_df.columns]
    salida = nuevos_df[columnas_id].copy().reset_index(drop=True)
    salida['score_predicho'] = pred_nuevas.astype('int8')
    salida['probabilidad_score_0'] = prob_nuevas[:, class_index[0]].round(4)
    salida['probabilidad_score_1'] = prob_nuevas[:, class_index[1]].round(4)
    display(salida.head(20))
    salida.to_csv('/content/predicciones_nuevos_datos.csv', index=False)
    print('Archivo guardado: /content/predicciones_nuevos_datos.csv')
else:
    print('Predicción por lote desactivada. Activa EJECUTAR_PREDICCION_REAL para usar un CSV nuevo.')
